In [62]:
import pandas as pd 
import duckdb as db
from pathlib import Path
import duckdb

# Construire un chemin RELATIF de manière sûre
db_path = Path(r"C:\Users\mhama\OneDrive\Documents\Projet_final\movies.duckdb")

# Vérifier si le fichier existe avant de se connecter
if not db_path.exists():
    raise FileNotFoundError(f"Base de données non trouvée à {db_path.resolve()}")

# Connexion
con = duckdb.connect(str(db_path))

# Tester la requête
tables = con.execute("SHOW TABLES;").fetchall()
tables

[('films',), ('filtered_ratings',), ('ratings',)]

In [21]:
films=tables[0][0]
films

df_ratings=con.execute("SELECT * FROM ratings LIMIT 100 ").df()
df_films=con.execute("SELECT * FROM films LIMIT 100").df()
print(df_films.columns,df_ratings.columns)


Index(['id', 'title', 'genres', 'description', 'release_date', 'vote_average',
       'vote_count'],
      dtype='object') Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')


In [23]:
df_films=df_films.rename(columns={"id" : "movieId"})

In [24]:
df=df_films.merge(df_ratings,on="movieId",how="right")[["userId","movieId","rating"]]

In [33]:
df_pivot=df.pivot(index="userId",columns="movieId",values="rating")
df_pivot

movieId,5,25,32,58,64,79,110,141,147,223,...,69844,73017,81834,91500,91542,92439,96821,98809,99114,112552
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,4.5,NaN,...,5.0,5.0,5.0,2.5,5.0,5.0,5.0,0.5,4.0,5.0
2,3.0,3.0,2.0,3.0,4.0,4.0,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
from sklearn.decomposition import TruncatedSVD

# Remplacer les NaN par 0 (important pour sklearn)
pivot_filled = df_pivot.fillna(0)

# Appliquer la SVD
svd = TruncatedSVD(n_components=50, random_state=42)
matrix_reduced = svd.fit_transform(pivot_filled)

print(matrix_reduced.shape)  # (nb_users, 50)


(4, 4)


In [41]:
from sklearn.decomposition import TruncatedSVD

# Remplacer les NaN par 0
pivot_filled = df_pivot.fillna(0)

# Appliquer SVD
svd = TruncatedSVD(n_components=50, random_state=42)
matrix_reduced = svd.fit_transform(pivot_filled)

# Reconstituer la matrice prédite
approximation = svd.inverse_transform(matrix_reduced)

# Remettre dans un DataFrame pour lire proprement
predicted_ratings = pd.DataFrame(approximation, index=df_pivot.index, columns=df_pivot.columns)

# Exemple : voir les notes prédites pour l'utilisateur 123
user_123_pred = predicted_ratings.loc[1]

# Trouver les films non encore notés par cet utilisateur
already_rated = df_pivot.loc[1][df_pivot.loc[1].notna()].index
recommendations = user_123_pred.drop(index=already_rated).sort_values(ascending=False)

print(recommendations.head(5))  # Top 5 films à recommander à user 123


movieId
25      1.785198e-15
5       1.131149e-15
1597   -2.763982e-17
1431   -2.763982e-17
2881   -2.763982e-17
Name: 1, dtype: float64
